# Transient chaos — phase-space Shannon entropy

Copy of `analysis/no_pump_frequency/phase_space_shannon_entropy copy.ipynb` (09/10/2026), with the parameters of the other
`poster_chicago_2026` notebooks:

- $\omega_a = 25.338776456203686$, fixed. The original drew it at random: `3.85*(rand(1)[1]/5+0.9)*2π`.
- $\kappa_b = \pi/10.4$, in the $-2\kappa_b\dot\phi_b$ convention. The original used $2/10.4$.
- $\varepsilon_d = 10.12\,g_2$ in both simulation loops, through `ϵ_d_g2_ratio`. The original parameter cell used 10.12, but the loops used 10.
- $\kappa_a = 0$, $\omega_d = \omega_b$, $\omega_p = 2\omega_a - \omega_d = 0$, same $\varphi_a, \varphi_b, E_J$ as before.
- The pump amplitude is called `ϵ_p` (it was `ϵ_0`). Unused variables (`Lamb_shift`, `κ_2`, the cell-level `ϵ_0`, `g2`, `ϵ_d`) are removed.

**12/10/2026:** $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$ (static pump), instead of $J_1(\varepsilon_p)E_J\varphi_a^2\varphi_b$, so $\varepsilon_d = 10.12\,g_2$ would be about twice as large as before at the same $\varepsilon_p$. There the chaos does not decay within /\kappa_b$. The ratio is therefore set to $\varepsilon_d/g_2 = 5.2$, the same physical drive as the former .12\,g_2^{J_1}$ at $\varepsilon_p = 0.802$ (/2\pi = 98.8$ MHz with the new definition).

The values $\varepsilon_p = 0.8$ and $0.802$ were chosen with the old $\kappa_b$. Times are in units of $1/\kappa_b$,
so $t_{max} = 2200/\kappa_b$ is about 1.6× shorter in ns than before.


In [ ]:
using ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

ϵ_d_g2_ratio = 5.2*2  # ϵ_d = ϵ_d_g2_ratio*g2. Same physical drive as the former 10.12 J1(ϵ_p)E_Jφ_a²φ_b at ϵ_p = 0.802 (10.12 J1/sin = 5.20), where the chaos is transient

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4 # 1/ns, convention φ̈_b = … − 2κ_b φ̇_b (db/dt = … − κ_b b for SystemATS_ladder)
κ_a = 0

In [ ]:
(sin(0.8)*E_J*φ_a^2*φ_b)/2/pi*1e3

In [ ]:
ϵ_p_list = [0.8]#range(start = 0.6044444444444445, stop = 0.802, length = 10);


t_max = 1500/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,10)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_p in ϵ_p_list
    g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b # static pump (ω_p = 0)
    ϵ_d=ϵ_d_g2_ratio*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Final_mode[(re_i,im_j)] = a
        end
    end
    Final_all[ϵ_p] = Final_mode
end

In [ ]:
for ϵ_p in ϵ_p_list

    g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b
    Final_mode = Final_all[ϵ_p]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=1, label="", color=:black)
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          top_margin=5mm)

    display(p0)
end


In [ ]:
ϵ_p_list = [0.802]#range(start = 0.6044444444444445, stop = 0.802, length = 10);


t_max = 6000/κ_b
t_min = 0.0
t_min_save = t_min
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,10)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_p in ϵ_p_list
    g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b # static pump (ω_p = 0)
    ϵ_d=ϵ_d_g2_ratio*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Final_mode[(re_i,im_j)] = a
        end
    end
    Final_all[ϵ_p] = Final_mode
end

In [ ]:
dict = Final_all[ϵ_p_list[1]]

# Phase-space resolution
nx = 200
ny = 200

# Collect all points once to define bounds
all_points = reduce(vcat, values(dict))
x_all = real.(all_points)
y_all = imag.(all_points)

x_edges = range(minimum(x_all), maximum(x_all); length=nx+1)
y_edges = range(minimum(y_all), maximum(y_all); length=ny+1)

In [ ]:
using StatsBase

function shannon_entropy(weights)
    p = weights ./ sum(weights)
    p = p[p .> 0]                 # avoid log(0)
    return -sum(p .* log.(p))
end
group_size = 200
trajectories = collect(values(dict))
ntraj = length(trajectories)

# number of chunks (use minimum length to stay safe)
nchunks = minimum(length(traj) ÷ group_size for traj in trajectories)

entropy_per_chunk = zeros(Float64, nchunks, ntraj)

for (j, traj) in enumerate(trajectories)
    for k in 1:nchunks
        i0 = (k-1)*group_size + 1
        i1 = k*group_size
        chunk = @view traj[i0:i1]

        h = fit(
            Histogram,
            (real.(chunk), imag.(chunk)),
            (x_edges, y_edges)
        )

        entropy_per_chunk[k, j] = shannon_entropy(h.weights)
    end
end

nbins_entropy = 500

Hmin = minimum(entropy_per_chunk)
Hmax = maximum(entropy_per_chunk)

entropy_bins = range(Hmin, Hmax; length=nbins_entropy+1)
entropy_centers = (entropy_bins[1:end-1] .+ entropy_bins[2:end]) ./ 2

entropy_hist_matrix = zeros(Float64, nbins_entropy, nchunks)

for k in 1:nchunks
    h = fit(Histogram, entropy_per_chunk[k, :], entropy_bins)
    entropy_hist_matrix[:, k] .= h.weights
end

chunk_indices = (0:nchunks-1) .* group_size .+ 1   # index of first point in chunk
chunk_time_converted = chunk_indices*dt_save*κ_b

In [ ]:

heatmap(
    chunk_time_converted,
    entropy_centers,
    entropy_hist_matrix;
    cmap = :viridis,
    xlabel = L"\kappa_bt",
    ylabel = "Shannon entropy",
    colorbar = true,
    #ylims=(5,7)
)

In [ ]:
using EasyFit
H_thresh_chaos = 5.0   # example threshold (choose physically meaningful value)
proportion_chaos = [count(>=(H_thresh_chaos), entropy_per_chunk[k, :]) for k in 1:nchunks-5]./length(Re_a_list)./length(Im_a_list)

H_thresh_reg = 5.0
proportion_reg = [count(<=(H_thresh_reg), entropy_per_chunk[k, :]) for k in 1:nchunks-5]./length(Re_a_list)./length(Im_a_list)

fit_exp_c = fitexp(chunk_time_converted[1:end-5],proportion_chaos)
fit_exp_r = fitexp(chunk_time_converted[1:end-5],proportion_reg)

A_chaos,B_chaos,C_chaos = fit_exp_c.a,fit_exp_c.b,fit_exp_c.c
A_reg,B_reg,C_reg = fit_exp_r.a,fit_exp_r.b,fit_exp_r.c

time_lis = 0:1:maximum(chunk_time_converted)
prop_chaos_lis =@. A_chaos*exp(-time_lis/B_chaos)+C_chaos
prop_reg_lis = @. A_reg*exp(-time_lis/B_reg)+C_reg;

In [ ]:
plot(
    chunk_time_converted[1:end-5]*2,
    proportion_chaos,
    lw = 2,
    label = "Chaotic traj."
)
plot!(
    time_lis*2,
    prop_chaos_lis,
    ls =:dash,
    label=""
)
plot!(
    chunk_time_converted[1:end-5]*2,
    proportion_reg,
    lw = 2,
    label = "Regular traj."
)
plot!(
    time_lis*2,
    prop_reg_lis,
    ls =:dash,
    label=""
)
plot!(
    xlabel = L"\kappa_bt",
    ylabel = "Fraction of trajectories",
#    ylims = (0.0, 1)
)

#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/torus_crisis/transient_chaos_12_10_2026.pdf")
#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/article_chaos/figures_article/transient_chaos_12_10_2026.pdf")

## Lifetime of the transient chaos against $\varepsilon_p$ (13/10/2026)

Scan of $\varepsilon_p$ from 0.66 to 1.10 ($g_2/2\pi \approx 84$–$122$ MHz) at the same $\varepsilon_d = 5.2\,g_2$ ($\varepsilon_d/2g_2 = 2.6$), with the protocol of the cells above, over $1000/\kappa_b$ (article convention, $\kappa_b/2\pi = 96$ MHz).

- **Escape time** of a trajectory: end of its last chaotic window. Only trajectories chaotic in at least one window count; those still chaotic in the last window are censored.
- **Survival** $S(t)$: fraction of these trajectories still chaotic after $t$. For a boundary crisis, $S(t) \approx e^{-t/\tau}$ and $\tau \propto (g_2 - g_{2,c})^{-\gamma}$ above the crisis.
- $\tau$ from an exponential fit of $S(t)$, cross-checked with the mean escape time of the uncensored trajectories (a lower bound when many are censored).
- Resolution: one window is 200 memory periods, i.e. $30/\kappa_b$ in the article convention. Shorter lifetimes are not resolved.

About 5 min on 10 threads.

In [ ]:
# Lifetime of the transient chaos against the pump amplitude, at fixed ϵ_d = ϵ_d_g2_ratio*g2 (cell 2).
# Same protocol as the cells above: 10×10 grid of initial memory states, b(0) = 0, one sample per memory period,
# windows of M = 200 periods, Shannon entropy on a 200×200 partition of the bounding box of all sampled points
# (one partition per ϵ_p), chaotic window if H ≥ H_c = 5.
# Times: κ_b here is the code rate (−2κ_b φ̇_b). In the article convention (κ_b/2π = 96 MHz), t_article = 2κ_b t.
using StatsBase

ϵ_p_scan = collect(0.66:0.02:1.0)      # g2/2π ≈ 84–122 MHz
t_max_scan = 500/κ_b                    # = 1000/κ_b in the article convention
dt_save_scan = 2π/ω_a                   # Poincaré map of the memory
t_save_scan = 0.0:dt_save_scan:t_max_scan
M_scan = 200
H_c_scan = 5.0
n_bins_scan = 200
Re_a_scan = range(-5, 5, 10)
Im_a_scan = range(-4, 4, 10)

function window_entropies(trajs, M, nbins)
    pts = reduce(vcat, trajs)
    x_edges = range(minimum(real.(pts)), maximum(real.(pts)); length=nbins+1)
    y_edges = range(minimum(imag.(pts)), maximum(imag.(pts)); length=nbins+1)
    K = minimum(length.(trajs)) ÷ M
    H = zeros(K, length(trajs))
    for (j, traj) in enumerate(trajs), k in 1:K
        chunk = @view traj[(k-1)*M+1:k*M]
        w = fit(Histogram, (real.(chunk), imag.(chunk)), (x_edges, y_edges)).weights
        q = w[w .> 0] ./ M
        H[k, j] = -sum(q .* log.(q))
    end
    return H
end

H_scan = Vector{Matrix{Float64}}(undef, length(ϵ_p_scan))   # one (window × trajectory) matrix per ϵ_p
Threads.@threads for i in eachindex(ϵ_p_scan)
    ϵ_p = ϵ_p_scan[i]
    g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b
    drive_prepared = prepare(DriveTwoModesBasics(ϵ_d_g2_ratio*g2, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase))
    trajs = Vector{Vector{ComplexF32}}()
    for x in Re_a_scan, y in Im_a_scan
        system_prepared = prepare(SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, x + 1im*y, 0.0))
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        u_0 = get_init(system_prepared, environment_prepared)
        a, _, _ = default_solver(u_0, (0.0, t_max_scan), p, dt, t_save = t_save_scan, equation = default_equation!)
        push!(trajs, ComplexF32.(a))
    end
    H_scan[i] = window_entropies(trajs, M_scan, n_bins_scan)
end

In [ ]:
# Survival of the chaos and lifetime τ against g2.
# Escape time of a trajectory: end of its last chaotic window. Only trajectories chaotic in at least one window count.
# Trajectories still chaotic in the last window are censored (escape time > t_max).
# S(t): fraction of these trajectories with escape time > t. τ: decay time of the fit S(t) ≈ A e^{-t/τ} + C (EasyFit),
# cross-checked with the mean escape time of the uncensored trajectories (a lower bound when many are censored).
using EasyFit

κ_art = 2κ_b                                                        # article convention
t_win = collect(1:size(H_scan[1], 1)) .* (M_scan*dt_save_scan*κ_art)   # κ_b t at the end of each window
g2_scan = sin.(ϵ_p_scan) .* E_J*φ_a^2*φ_b/2π*1e3                     # MHz

τ_fit  = fill(NaN, length(ϵ_p_scan))
τ_mean = fill(NaN, length(ϵ_p_scan))
n_chaotic = zeros(Int, length(ϵ_p_scan))
n_censored = zeros(Int, length(ϵ_p_scan))
p_surv = plot(xlabel=L"\kappa_b t", ylabel="Surviving chaotic fraction", legend=:outerright, ylims=(1e-2, 1.1))
for i in eachindex(ϵ_p_scan)
    chaotic = H_scan[i] .>= H_c_scan
    ever = findall(vec(any(chaotic, dims=1)))
    n_chaotic[i] = length(ever)
    n_chaotic[i] < 5 && continue
    last_win = [findlast(chaotic[:, j]) for j in ever]
    t_esc = t_win[last_win]
    censored = last_win .== length(t_win)
    n_censored[i] = count(censored)
    S = [count(>(t), t_esc)/length(t_esc) for t in t_win]
    any(.!censored) && (τ_mean[i] = sum(t_esc[.!censored])/count(.!censored))
    if n_censored[i] < n_chaotic[i]
        try
            τ_fit[i] = fitexp(t_win, S).b
        catch err
            @warn "fit failed at ϵ_p = $(ϵ_p_scan[i])" err
        end
    end
    plot!(p_surv, t_win, max.(S, 1e-2), label=string(round(g2_scan[i], digits=1))*" MHz")
end
display(p_surv)

p_tau = scatter(g2_scan, τ_fit, xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\kappa_b\tau",
                label="exponential fit of S(t)", legend=:topright)
scatter!(p_tau, g2_scan, τ_mean, marker=:diamond, label="mean escape time (uncensored)")
hline!(p_tau, [t_win[end]], ls=:dash, color=:gray, label=L"\kappa_b t_{max}")
display(p_tau)

for i in eachindex(ϵ_p_scan)
    println(rpad(round(g2_scan[i], digits=1), 7), " MHz  chaotic ", lpad(n_chaotic[i], 3), "  censored ", lpad(n_censored[i], 3),
            "  κ_b τ (fit) ", round(τ_fit[i], digits=1), "  κ_b τ (mean) ", round(τ_mean[i], digits=1))
end